<a href="https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TemitopeAlawode/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## **Setup**

In [9]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### **Task Type: Signal Analysis (primary) + Ranking (secondary)**


My project is primarily **signal analysis** - I'm investigating which observed signals (position, CTR, engagement, content age, search volume, etc.) are associated with page performance.

**Secondary task:** The analysis will support **ranking** - identifying which pages are most at risk or have the most opportunity, based on the signals we find.

**Why not classification?** I'm not predicting a single yes/no outcome. Instead, I'm exploring relationships and quantifying which signals matter most. This feeds into a ranking system for content recommendations.

**Why signal analysis?** From the skill file: *"Which signals travel together?"* is the signal analysis task type. My Week 1 findings showed surprising patterns (search volume correlation = 0.001, engagement inverse relationship with impressions) - this tells me there are real signals to discover.

In [10]:
# This cell shows the task type in action

import pandas as pd
import numpy as np

# Load data (after setup)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=== Task Type: Signal Analysis ===\n")
print(f"Total pages: {len(df):,}")
print(f"Number of potential signals: {len(df.columns)}\n")

# Show that we're analyzing relationships between signals
signals = ['impressions_90d', 'ctr', 'avg_position', 'engagement_rate',
           'content_age_days', 'search_volume', 'word_count']

print("Example: How signals relate to performance")
print("Correlation with impressions_90d:\n")
for signal in signals:
    corr = df[signal].corr(df['impressions_90d'])
    print(f"  {signal}: {corr:.3f}")

print("\n→ Some signals correlate, some don't. Signal analysis helps us understand which matter.")

=== Task Type: Signal Analysis ===

Total pages: 30,000
Number of potential signals: 44

Example: How signals relate to performance
Correlation with impressions_90d:

  impressions_90d: 1.000
  ctr: -0.019
  avg_position: -0.071
  engagement_rate: 0.024
  content_age_days: -0.001
  search_volume: 0.001
  word_count: 0.163

→ Some signals correlate, some don't. Signal analysis helps us understand which matter.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### **Target: observed performance measures**

**Primary Target (Proxy):** `impressions_90d` - a measure of page visibility

**Secondary Target:** `is_declining_label` - whether a page's trend is down (derived from `trend_direction`)

**Why these targets?**

- `impressions_90d` is a direct observable measure of visibility

- `is_declining_label` is derived from `trend_direction`, which is calculated from `trend_pct`

- Both are observed in the data, not defined by a product rule

**⚠ Important: The label trap**

According to the FlyRank data skill file:

*"`is_declining_label` is derived from `trend_direction`, which is computed from `trend_pct`. Therefore `trend_direction` and `trend_pct` are NEVER features."*

**What this means:** When I use `is_declining_label` as a target, I must NEVER use `trend_direction` or `trend_pct` as features. That would be leakage - the answer would already be in the feature!

In [11]:
# Show the targets

print("=== Target/Proxy Examples ===\n")

# Target 1: impressions_90d (visibility)
print("Target 1: impressions_90d (visibility measure)")
print(df['impressions_90d'].describe().round(0))
print()

# Target 2: is_declining_label (performance trajectory)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print("Target 2: is_declining_label (performance trajectory)")
print(f"Declining: {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean()*100:.1f}%)")
print(f"Not declining: {len(df) - df['is_declining_label'].sum():,} ({(1 - df['is_declining_label'].mean())*100:.1f}%)")
print()

print("⚠️ Remember: trend_direction and trend_pct are NEVER features!")
print("The label comes from trend_direction, so those columns would leak the answer.")

=== Target/Proxy Examples ===

Target 1: impressions_90d (visibility measure)
count     30000.0
mean       5200.0
std       16838.0
min           1.0
25%          81.0
50%         731.0
75%        3615.0
max      517715.0
Name: impressions_90d, dtype: float64

Target 2: is_declining_label (performance trajectory)
Declining: 16,262 (54.2%)
Not declining: 13,738 (45.8%)

⚠️ Remember: trend_direction and trend_pct are NEVER features!
The label comes from trend_direction, so those columns would leak the answer.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### **Success Metrics**

**For signal analysis:**

- **Effect sizes** - how big is the relationship? (e.g., correlation coefficient, mean differences)

- **Grouped comparisons** - how do signals differ across performance groups?

- **Feature importance** - which signals rank highest in explaining variation?

**For ranking (later):**

- **Precision@K** - of the top K pages flagged, how many are actually high-performing or declining?

- **Average precision** - how good is the overall ranking?

**Why these metrics:**

- Effect sizes tell us **how much** a signal matters (not just "it matters")

- Precision@K matches the **real-world decision@** (editor reviews top K pages)

- These metrics are **interpretable** - I can explain them to a non-technical stakeholder

In [12]:
# Show the metrics in action


print("=== Success Metrics ===\n")

# Metric 1: Effect size - how much does CTR differ between high and low performers?
df['high_performance'] = (df['impressions_90d'] > df['impressions_90d'].median()).astype(int)
high_ctr = df[df['high_performance'] == 1]['ctr'].mean()
low_ctr = df[df['high_performance'] == 0]['ctr'].mean()

print("Metric 1: Effect size (CTR difference between groups)")
print(f"High performers: {high_ctr:.3f}")
print(f"Low performers:  {low_ctr:.3f}")
print(f"Difference:      {high_ctr - low_ctr:.3f}")
print("→ This tells us HOW MUCH CTR matters\n")

# Metric 2: Correlation - how strong is the relationship?
corr = df['ctr'].corr(df['impressions_90d'])
print("Metric 2: Correlation between CTR and impressions")
print(f"Correlation: {corr:.3f}")
print("→ This tells us the STRENGTH of the relationship\n")

# Metric 3: Precision@K (example - for ranking)
# Sort by impressions and check top 20 are actually high performers
order = np.argsort(-df['impressions_90d'].values)
top20 = df.iloc[order[:20]]['high_performance'].mean()
print("Metric 3: Precision@20 (for ranking)")
print(f"Precision@20: {top20:.3f}")
print(f"→ Of top 20 pages by impressions, {top20*20:.0f} are 'high performers'")
print("\nThese metrics help us measure what 'good' means for our analysis.")

=== Success Metrics ===

Metric 1: Effect size (CTR difference between groups)
High performers: 0.270
Low performers:  0.752
Difference:      -0.482
→ This tells us HOW MUCH CTR matters

Metric 2: Correlation between CTR and impressions
Correlation: -0.019
→ This tells us the STRENGTH of the relationship

Metric 3: Precision@20 (for ranking)
Precision@20: 1.000
→ Of top 20 pages by impressions, 20 are 'high performers'

These metrics help us measure what 'good' means for our analysis.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### **Unit of Analysis: One row = one content page**

Each row in the dataset represents **a single content page** (a specific URL/article).

**Why this matters:**

- Everything we analyze is at the page level

- We're looking at what makes individual pages succeed or fail

- Our recommendations will be page-specific

**Check:** Does the data have one row per page?

- `content_id` is the unique identifier for each page

- Number of unique content_ids = total rows ✅

In [13]:
# Show the unit of analysis

print("=== Unit of Analysis ===\n")
print("One row = one content page")
print(f"Total rows (pages): {len(df):,}")
print(f"Total clients: {df['client_id'].nunique():,}")
print(f"One row per content_id? {df['content_id'].nunique() == len(df)} ✅\n")

# Show sample rows
sample_cols = ['content_id', 'client_id', 'impressions_90d', 'ctr', 'avg_position',
               'engagement_rate', 'content_age_days', 'trend_direction']

print("Sample rows (each row is ONE page):")
df[sample_cols].head(10)

=== Unit of Analysis ===

One row = one content page
Total rows (pages): 30,000
Total clients: 32
One row per content_id? True ✅

Sample rows (each row is ONE page):


,content_id,client_id,impressions_90d,ctr,avg_position,engagement_rate,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,0.76,10.6,5.88,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,0.05,20.3,0.00,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,0.09,36.5,0.00,141,down
3,content_331d6c4de07b,client_19581e27de,11751,0.49,6.2,1.28,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,0.13,44.0,0.00,263,down
5,content_d4084a4bc775,client_f369cb89fc,3970,0.03,8.5,0.00,147,down
6,content_9a34b442b552,client_8722616204,20,0.00,7.0,0.00,90,down
7,content_a63219c6e95a,client_19581e27de,1724,0.06,21.2,3.57,445,stable
8,content_5e6c160719bc,client_6208ef0f77,32574,0.09,46.0,5.88,90,down
9,content_c27558df2b0c,client_19581e27de,1240,0.16,4.9,0.00,257,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### **Why ML/Data Analysis Beats a Fixed Rule**

**A fixed rule isn't enough because:**

1. **Multiple interacting signals -** Position, CTR, engagement, age, and content type may all interact. A rule like "if CTR > 0.5 and position < 10 then good" misses these interactions.

2. **Relationships aren't linear -** The relationship between position and CTR isn't simple. Dropping from position 1 to 2 hurts more than dropping from 10 to 11.

3. **Assumptions are often wrong -** My Week 1 analysis showed: search volume doesn't predict traffic (correlation 0.001), and engagement has an inverse relationship with impressions. Fixed rules based on assumptions would fail here.

4. **Too many signals to combine by hand -** With 44 columns of potential signals, finding the optimal combination manually is nearly impossible.

5. **Signal weights shift over time -** What matters today might change next quarter. A fixed rule would need constant manual updates.

**What ML/analysis gives us:**

- **Objective weighting -** Let the data reveal which signals matter most

- **Interaction discovery -** Find combinations of signals that predict performance

- **Honest validation -** Test the pattern on unseen data

- **Adaptability -** Update as patterns shift

In [14]:
# Show why a simple rule fails

print("=== Why a Fixed Rule Fails ===\n")

# Example: A simple rule might say "pages with high CTR are good"
# But let's test that assumption

# Create a simple rule: if ctr > 0.3, call it "good"
df['simple_rule_good'] = (df['ctr'] > 0.3).astype(int)

# But what if we check actual performance?
df['actual_good'] = (df['impressions_90d'] > df['impressions_90d'].median()).astype(int)

# Compare
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(df['actual_good'], df['simple_rule_good'])
precision = (df[df['simple_rule_good'] == 1]['actual_good'].sum() /
             df[df['simple_rule_good'] == 1].shape[0])

print("Simple rule: 'If CTR > 0.3, page is good'")
print(f"Accuracy of this simple rule: {accuracy:.3f}")
print(f"Precision (when rule says 'good', is it actually good?): {precision:.3f}")
print("\nA simple rule is OK, but ML can find better patterns.")
print("ML can find combinations of signals that work better together.\n")

# Show the problem with simple rules
print("Example: The rule would miss these cases:")
problem_cases = df[(df['ctr'] < 0.3) & (df['actual_good'] == 1)].head(3)
print(problem_cases[['content_id', 'ctr', 'impressions_90d']])
print("\nThese pages have low CTR but are actually high performers.")
print("A fixed rule would ignore them. ML can find more nuanced patterns.")

=== Why a Fixed Rule Fails ===

Simple rule: 'If CTR > 0.3, page is good'
Accuracy of this simple rule: 0.563
Precision (when rule says 'good', is it actually good?): 0.633

A simple rule is OK, but ML can find better patterns.
ML can find combinations of signals that work better together.

Example: The rule would miss these cases:
             content_id   ctr  impressions_90d
1  content_a1fb4e703a9e  0.05            15320
2  content_9aa793d4d895  0.09            12581
4  content_d99b7a2d90ca  0.13            19140

These pages have low CTR but are actually high performers.
A fixed rule would ignore them. ML can find more nuanced patterns.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.